# RAG Impelentation


## Install libraries

In [ ]:
!pip install groq sentence-transformers faiss-cpu pypdf numpy matplotlib scikit-learn -q

## Import libraries


In [ ]:
import os
import numpy as np
import faiss

from groq import Groq
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader

## API KEY


In [ ]:
from getpass import getpass

GROQ_API_KEY = getpass("Enter your Groq API Key: ")

## Setup Groq API

In [ ]:
client = Groq(
    api_key=GROQ_API_KEY
)

MODEL = "llama-3.1-8b-instant"

## Ask LLM without RAG


In [ ]:
messages = [
    {
        "role": "user",
        "content": "What is inside my private PDF?"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages
)

print(response.choices[0].message.content)

## Load embedding model


In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

## Create documents	

In [ ]:
documents = [
    "RAG stands for Retrieval Augmented Generation.",
    "Embeddings convert text into vectors.",
    "FAISS is used for similarity search.",
    "Chunking splits large documents into smaller pieces."
]

print(documents)

## Generate embeddings

In [ ]:
embeddings = embedding_model.encode(documents)

print(embeddings.shape)

## Create FAISS index without clustering


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings, dtype="float32")
)

print(index.ntotal)

## Create FAISS index with clustering

In [ ]:
dimension = embeddings.shape[1]

nlist = 2

quantizer = faiss.IndexFlatL2(dimension)

index = faiss.IndexIVFFlat(
    quantizer,
    dimension,
    nlist,
    faiss.METRIC_L2
)

index.train(np.array(embeddings, dtype="float32"))

index.add(np.array(embeddings, dtype="float32"))

print("Total vectors stored:", index.ntotal)

## Ask a question

In [ ]:
query = "What does RAG stand for?"

query_embedding = embedding_model.encode([query])

## Search similar chunks


In [ ]:
distances, indices = index.search(
    np.array(query_embedding, dtype="float32"),
    2
)

print(indices)

## Retrieve matching docs

In [ ]:
retrieved_docs = [
    documents[i]
    for i in indices[0]
]

print(retrieved_docs)

## Create context(joining)

In [ ]:
context = "\n".join(retrieved_docs)

print(context)

## Send context to LLM

In [ ]:
messages = [
    {
        "role": "system",
        "content": "Answer only using the provided context."
    },
    {
        "role": "user",
        "content": f"Context:\n{context}\n\nQuestion: {query}"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages
)

print(response.choices[0].message.content)

# NOW FOR PDF

## Load PDF File

In [ ]:
pdf = PdfReader(r"C:\Users\DELL\Downloads\Kj.pdf")

text = ""

for page in pdf.pages:
    text += page.extract_text()

print(text[:1000])

## Chunk the PDF Text

In [ ]:
chunk_size = 300

chunks = []

for i in range(0, len(text), chunk_size):
    chunk = text[i:i + chunk_size]
    chunks.append(chunk)

print("Total Chunks:", len(chunks))

## Generate Chunk Embeddings

In [ ]:
chunk_embeddings = embedding_model.encode(chunks)

print(chunk_embeddings.shape)

## Store Chunks in FAISS

In [ ]:
dimension = chunk_embeddings.shape[1]

pdf_index = faiss.IndexFlatL2(dimension)

pdf_index.add(
    np.array(chunk_embeddings, dtype="float32")
)

print(pdf_index.ntotal)

## Ask Questions on PDF

In [ ]:
query = "What is the document about?"

query_embedding = embedding_model.encode([query])

## Retrieve Relevant PDF Chunks

In [ ]:
distances, indices = pdf_index.search(
    np.array(query_embedding, dtype="float32"),
    3
)

retrieved_chunks = [
    chunks[i]
    for i in indices[0]
]

print(retrieved_chunks)

## Generate Final RAG Answer

In [ ]:
context = "\n".join(retrieved_chunks)

messages = [
    {
        "role": "system",
        "content": "Answer only from the provided context."
    },
    {
        "role": "user",
        "content": f"Context:\n{context}\n\nQuestion: {query}"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages
)

print(response.choices[0].message.content)